# 14 - Ragas Evaluation

This notebook explores Ragas quality metrics for RAG outputs and compares local experimentation with the canonical Atlas Backend runtime endpoint at `/api/rag/evaluate`. Keep live evaluator calls opt-in, because they require a configured LiteLLM model and may consume tokens.

In [ ]:
import json
import os

import httpx

SAMPLE_RECORD = {
    "question": "What service routes model calls in Atlas?",
    "answer": "Atlas routes model calls through LiteLLM.",
    "contexts": [
        "LiteLLM is the always-on OpenAI-compatible gateway for every LLM provider in Atlas."
    ],
    "ground_truth": "Atlas routes model calls through LiteLLM.",
}

METRICS = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]

## 1. Optional Local Ragas Evaluation

Set `RUN_RAGAS_LIVE=1` to run local Ragas evaluation from the notebook image. The cell uses the same LiteLLM environment variables that Atlas injects into JupyterHub.

In [ ]:
if os.getenv("RUN_RAGAS_LIVE") == "1":
    import asyncio
    import concurrent.futures
    import inspect

    from openai import AsyncOpenAI
    from ragas.embeddings import OpenAIEmbeddings
    from ragas.llms import llm_factory
    from ragas.metrics.collections import (
        AnswerRelevancy,
        ContextPrecision,
        ContextRecall,
        Faithfulness,
    )

    base_url = os.getenv("OPENAI_API_BASE", os.getenv("LITELLM_BASE_URL", "http://litellm:4000") + "/v1")
    api_key = os.getenv("OPENAI_API_KEY", os.getenv("LITELLM_API_KEY", ""))
    evaluator_model = os.getenv("RAGAS_EVALUATOR_MODEL", os.getenv("LITELLM_DEFAULT_MODEL", "ollama/qwen3.6:latest"))
    embeddings_model = os.getenv("RAGAS_EMBEDDINGS_MODEL", os.getenv("LITELLM_EMBEDDING_MODEL", "ollama/nomic-embed-text"))

    # ragas 0.4.3's ragas.metrics.collections metrics are the modern surface:
    # the answer-relevancy class is AnswerRelevancy (ResponseRelevancy does not
    # exist in 0.4.3), they take the llm/embeddings at construction, require an
    # InstructorLLM built via llm_factory (not LangchainLLMWrapper), and are
    # scored through their async abatch_score API rather than the deprecated
    # top-level ragas.evaluate(). This mirrors the backend /api/rag/evaluate
    # implementation (rag_eval_service.py, issue #596).
    record = {
        "user_input": SAMPLE_RECORD["question"],
        "response": SAMPLE_RECORD["answer"],
        "retrieved_contexts": SAMPLE_RECORD["contexts"],
        "reference": SAMPLE_RECORD["ground_truth"],
    }

    async def _evaluate():
        async with AsyncOpenAI(api_key=api_key, base_url=base_url) as client:
            llm = llm_factory(evaluator_model, client=client)
            embeddings = OpenAIEmbeddings(client=client, model=embeddings_model)
            metric_objects = [
                Faithfulness(llm=llm),
                AnswerRelevancy(llm=llm, embeddings=embeddings),
                ContextPrecision(llm=llm),
                ContextRecall(llm=llm),
            ]
            scores = {}
            for metric in metric_objects:
                accepted = set(inspect.signature(metric.ascore).parameters)
                inputs = {key: value for key, value in record.items() if key in accepted}
                result = (await metric.abatch_score([inputs]))[0]
                scores[metric.name] = getattr(result, "value", None)
            return scores

    # Run on a dedicated thread so asyncio.run() never collides with the
    # notebook kernel's own running event loop.
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
        result = pool.submit(lambda: asyncio.run(_evaluate())).result()
    for name, value in result.items():
        print(f"{name}: {value}")
else:
    print("Set RUN_RAGAS_LIVE=1 to run local Ragas evaluation through LiteLLM.")

## 2. Backend Runtime Endpoint

Use `POST /api/rag/evaluate` when a notebook, n8n workflow, or service needs the shared Atlas Ragas contract. The endpoint accepts records with `question`, `answer`, `contexts`, and optional `ground_truth`, then returns metric scores per record.

In [ ]:
backend_url = os.getenv("BACKEND_API_URL", "http://backend:8000").rstrip("/")
backend_token = os.getenv("BACKEND_NOTEBOOK_API_TOKEN")
backend_headers = {"Authorization": f"Bearer {backend_token}"} if backend_token else {}
payload = {
    "records": [SAMPLE_RECORD],
    "metrics": METRICS,
}

try:
    response = httpx.post(f"{backend_url}/api/rag/evaluate", headers=backend_headers, json=payload, timeout=120.0)
    response.raise_for_status()
    print(json.dumps(response.json(), indent=2))
except httpx.HTTPError as exc:
    print(f"Backend Ragas evaluation request failed: {exc}")
    print("Confirm BACKEND_SOURCE=container, LiteLLM is healthy, and an evaluator model is configured.")

## 3. Workflow Notes

- Keep local notebook evaluation opt-in while selecting metrics and representative records.
- Promote production scoring workflows to the Backend `/api/rag/evaluate` endpoint.
- n8n and future ingestion jobs should call the Backend endpoint rather than adding their own Ragas dependency.
- Use `ground_truth` whenever running `context_precision` or `context_recall`.